In [1]:
# ============================================
# InDesign Audit Tool - Jupyter Notebook
# ============================================

# --- 1. Setup ---
# Install required packages (run in a notebook cell)
# !pip install lxml openai pandas

import zipfile
import xml.etree.ElementTree as ET
import pandas as pd
from openai import OpenAI

# Initialize OpenAI client (make sure OPENAI_API_KEY is set in your environment)
client = OpenAI()

# --- 2. Extract IDML Content ---
def extract_idml_content(file_path):
    """Extract fonts, links, and text from an IDML file."""
    fonts, links, texts = [], [], []
    
    with zipfile.ZipFile(file_path, 'r') as z:
        for name in z.namelist():
            if name.endswith('.xml'):
                data = z.read(name)
                root = ET.fromstring(data)
                
                # Collect fonts
                for elem in root.findall(".//Font"):
                    fonts.append(elem.attrib)
                
                # Collect links (images, assets)
                for elem in root.findall(".//Link"):
                    links.append(elem.attrib)
                
                # Collect text content
                for elem in root.findall(".//Story"):
                    texts.append(ET.tostring(elem, encoding="unicode"))
    
    return fonts, links, texts

# Example usage:
# fonts, links, texts = extract_idml_content("sample.idml")

# --- 3. AI-Powered Audit ---
def audit_text_with_ai(text):
    """Send text to OpenAI for readability and consistency audit."""
    response = client.chat.completions.create(
        model="gpt-4.1",
        messages=[
            {"role":"system","content":"You are an InDesign audit assistant."},
            {"role":"user","content":f"Audit this text for clarity, tone, and consistency:\n{text}"}
        ]
    )
    return response.choices[0].message.content

# Example usage:
# ai_feedback = audit_text_with_ai(texts[0])
# print(ai_feedback)

# --- 4. Rule-Based Audit ---
def rule_based_audit(fonts, links):
    """Check for missing fonts and broken links."""
    issues = []
    
    # Example: flag missing fonts
    for f in fonts:
        if "Status" in f and f["Status"] == "Missing":
            issues.append({"issue":"Missing font", "details":f})
    
    # Example: flag broken links
    for l in links:
        if "LinkResourceURI" in l and "file://" not in l["LinkResourceURI"]:
            issues.append({"issue":"Broken link", "details":l})
    
    return issues

# Example usage:
# issues = rule_based_audit(fonts, links)

# --- 5. Reporting ---
def generate_report(issues, ai_feedbacks):
    """Create a structured report of audit results."""
    df = pd.DataFrame(issues)
    print("=== Technical Issues ===")
    display(df)
    
    print("\n=== AI Feedback ===")
    for i, fb in enumerate(ai_feedbacks):
        print(f"\nText Block {i+1}:\n{fb}")

# Example usage:
# ai_feedbacks = [audit_text_with_ai(t) for t in texts[:2]]
# generate_report(issues, ai_feedbacks)

# ============================================
# End of Notebook Template
# ============================================

